# Phase 2: Single-Modality Models

Phase 2 builds standalone text and image pipelines with QWK evaluation. Text has an immediately runnable TF-IDF + SVM baseline and optional BERT embeddings. Image uses a ResNet50 transfer-learning pipeline.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src.data import get_paths, load_train_data, make_stratified_split
from src.image_model import (
    PetImageClassificationDataset,
    build_image_transform,
    build_resnet50_classifier,
    image_presence_summary,
)
from src.text_model import (
    TransformerTextConfig,
    TransformerTextEmbedder,
    build_tfidf_svm_classifier,
    descriptions_from_frame,
)
from src.train import TorchClassifierTrainer, cross_validate_text_classifier
from src.utils import classification_metrics, ensure_dir, seed_everything

seed_everything(42)
paths = get_paths(root=PROJECT_ROOT)
train_df = load_train_data(paths)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Train shape: {train_df.shape}")
print(f"Device: {device}")

Train shape: (14993, 24)
Device: cuda


## Run Configuration

The text baseline runs quickly on CPU. Image and BERT runs are heavier, so they are controlled by flags.

In [2]:
RUN_TEXT_TFIDF_CV = True
RUN_BERT_EMBEDDINGS = True
RUN_IMAGE_TRAINING = True

# Keep image training small while iterating; set to None for all rows after the pipeline works.
IMAGE_SAMPLE_SIZE = 1024
IMAGE_BATCH_SIZE = 32
IMAGE_EPOCHS = 3
USE_PRETRAINED_RESNET = True

feature_dir = ensure_dir(paths.outputs_dir / "features")
report_dir = ensure_dir(paths.outputs_dir / "reports")
checkpoint_dir = ensure_dir(paths.outputs_dir / "checkpoints")

## Text-Only Baseline: TF-IDF + Linear SVM

This is the simple text baseline requested in Phase 2. It gives a real text-only reference point before fine-tuning transformers.

In [3]:
texts = descriptions_from_frame(train_df)
labels = train_df["AdoptionSpeed"].astype(int).to_numpy()

if RUN_TEXT_TFIDF_CV:
    text_model = build_tfidf_svm_classifier(max_features=50000)
    text_cv = cross_validate_text_classifier(text_model, texts, labels, n_splits=5, random_state=42)
    text_results = pd.DataFrame([
        {
            "model": "TF-IDF + LinearSVC",
            "mean_qwk": text_cv["mean_qwk"],
            "std_qwk": text_cv["std_qwk"],
            "fold_scores": text_cv["fold_scores"],
        }
    ])
    display(text_results)
    text_results.to_csv(report_dir / "text_tfidf_svm_cv.csv", index=False)
else:
    print("Set RUN_TEXT_TFIDF_CV=True to run the text baseline.")

Fold 1/5 QWK: 0.2410
Fold 2/5 QWK: 0.2380
Fold 3/5 QWK: 0.2063
Fold 4/5 QWK: 0.2315
Fold 5/5 QWK: 0.2746


,model,mean_qwk,std_qwk,fold_scores
0,TF-IDF + LinearSVC,0.238295,0.021891,"[0.24097948731159202, 0.23799566464278044, 0.2..."


## Optional Transformer Text Embeddings

The text pipeline uses BERT-family description embeddings. This cell caches CLS embeddings by `PetID` for later fusion.

In [4]:
if RUN_BERT_EMBEDDINGS:
    cache_path = feature_dir / "bert_description_train.npz"
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=True)
        print(f"Using cached BERT embeddings: {cache_path}")
        print(cached["embeddings"].shape)
    else:
        config = TransformerTextConfig(
            model_name="bert-base-multilingual-cased",
            max_length=160,
            batch_size=16,
        )
        embedder = TransformerTextEmbedder(config=config, device=device)
        embeddings = embedder.encode(texts)
        np.savez_compressed(
            cache_path,
            pet_ids=train_df["PetID"].astype(str).to_numpy(),
            embeddings=embeddings,
        )
        print(embeddings.shape)
else:
    print("Set RUN_BERT_EMBEDDINGS=True after installing transformers and downloading model weights.")

Using cached BERT embeddings: C:\Users\andys\Documents\4662_Project\outputs\features\bert_description_train.npz
(14993, 768)


## Image Data Check

The image pipeline uses `Data/train_images` for train IDs. Missing images are handled with a blank-image fallback so DataLoader batches remain stable.

In [5]:
image_summary = image_presence_summary(train_df["PetID"], paths.train_images_dir)
image_summary

{'pets': 14993,
 'pets_with_images': 14652,
 'pets_without_images': 341,
 'total_images': 58311,
 'mean_images_per_pet': 3.889214966984593}

## Image-Only ResNet50 Classifier

This trains a frozen-backbone ResNet50 head first.

In [6]:
if RUN_IMAGE_TRAINING:
    image_df = train_df
    if IMAGE_SAMPLE_SIZE is not None and IMAGE_SAMPLE_SIZE < len(train_df):
        _, image_df = train_test_split(
            train_df,
            test_size=IMAGE_SAMPLE_SIZE,
            stratify=train_df["AdoptionSpeed"],
            random_state=42,
        )
        image_df = image_df.reset_index(drop=True)

    image_train_df, image_val_df = make_stratified_split(
        image_df,
        test_size=0.2,
        random_state=42,
    )

    train_dataset = PetImageClassificationDataset(
        image_train_df,
        image_dir=paths.train_images_dir,
        transform=build_image_transform(train=True),
    )
    val_dataset = PetImageClassificationDataset(
        image_val_df,
        image_dir=paths.train_images_dir,
        transform=build_image_transform(train=False),
    )
    train_loader = DataLoader(train_dataset, batch_size=IMAGE_BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=IMAGE_BATCH_SIZE, shuffle=False, num_workers=0)

    image_model = build_resnet50_classifier(
        num_classes=5,
        pretrained=USE_PRETRAINED_RESNET,
        freeze_backbone=True,
    )
    trainer = TorchClassifierTrainer(
        image_model,
        device=device,
        labels=image_train_df["AdoptionSpeed"].to_numpy(),
        learning_rate=1e-3,
        checkpoint_dir=checkpoint_dir,
        checkpoint_name="resnet50_image_only_best.pt",
    )
    image_history = trainer.fit(train_loader, val_loader, epochs=IMAGE_EPOCHS, patience=2)
    image_history_df = pd.DataFrame(image_history)
    display(image_history_df)
    image_history_df.to_csv(report_dir / "image_resnet50_history.csv", index=False)
else:
    print("Set RUN_IMAGE_TRAINING=True to train the image-only ResNet50 model.")

,epoch,train_loss,val_loss,val_qwk
0,1.0,1.624805,1.603849,0.093631
1,2.0,1.488408,1.598692,0.137288
2,3.0,1.424933,1.584781,0.180971
